# Lab | Data Structuring and Combining Data

This notebook combines and cleans the insurance customer files, then uses pivot tables to structure the marketing data.

## Challenge 1: Combining and cleaning data

The three files describe the same type of insurance customer data, but file3 uses State and Gender instead of ST and GENDER. Files 1 and 2 also contain percentage-formatted lifetime values and date-formatted complaint counts.

In [ ]:
import pandas as pd

base_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/"
file_urls = [base_url + filename for filename in ["file1.csv", "file2.csv", "file3.csv"]]

common_columns = [
    "Customer", "ST", "GENDER", "Education", "Customer Lifetime Value",
    "Income", "Monthly Premium Auto", "Number of Open Complaints",
    "Total Claim Amount", "Policy Type", "Vehicle Class"
]


def clean_customer_file(data):
    data = data.copy()
    data = data.rename(columns={"State": "ST", "Gender": "GENDER"})
    data = data.dropna(how="all")

    # Convert percentage-formatted values to their decimal equivalent.
    lifetime_text = data["Customer Lifetime Value"].astype("string")
    has_percent_sign = lifetime_text.str.contains("%", na=False)
    data["Customer Lifetime Value"] = pd.to_numeric(
        lifetime_text.str.replace("%", "", regex=False), errors="coerce"
    )
    data.loc[has_percent_sign, "Customer Lifetime Value"] /= 100

    # Convert both 0/1/2 and date-like values such as 1/2/00 to integers.
    complaints_text = data["Number of Open Complaints"].astype("string")
    complaints_as_numbers = pd.to_numeric(complaints_text, errors="coerce")
    complaints_as_dates = pd.to_numeric(
        complaints_text.str.extract(r"^1/(\d+)/", expand=False), errors="coerce"
    )
    data["Number of Open Complaints"] = complaints_as_numbers.fillna(
        complaints_as_dates
    )

    numeric_columns = [
        "Income", "Monthly Premium Auto", "Total Claim Amount"
    ]
    for column in numeric_columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

    return data[common_columns]


clean_files = [clean_customer_file(pd.read_csv(url)) for url in file_urls]
combined_data = pd.concat(clean_files, ignore_index=True)
# Keep the final version when the same customer appears in more than one file.
combined_data = (
    combined_data.drop_duplicates(subset="Customer", keep="last")
    .reset_index(drop=True)
)

print("Rows in each cleaned file:", [len(data) for data in clean_files])
print("Combined dimensions:", combined_data.shape)
print("\nCombined columns:", combined_data.columns.tolist())
print("\nMissing values:\n", combined_data.isna().sum())

The files are now aligned to one schema, malformed numeric values are converted, blank rows are removed, and repeated customer IDs are kept only once. Keeping the last version means that the clean records from file3 replace earlier dirty versions when the same customer appears in multiple files.

## Challenge 2: Structuring data

In [ ]:
marketing_url = (
    base_url + "marketing_customer_analysis_clean.csv"
)
marketing_data = pd.read_csv(marketing_url)

# Remove the technical index column because it is not an analysis variable.
marketing_data = marketing_data.drop(columns=["unnamed:_0"])
marketing_data["effective_to_date"] = pd.to_datetime(
    marketing_data["effective_to_date"]
)

print("Marketing data dimensions:", marketing_data.shape)
print("\nData types:\n", marketing_data.dtypes)

### Exercise 1: Revenue by sales channel

The dataset does not contain a column literally named revenue. Customer Lifetime Value is used as the available revenue proxy, and the pivot table sums it by sales channel.

In [ ]:
revenue_by_sales_channel = marketing_data.pivot_table(
    index="sales_channel",
    values="customer_lifetime_value",
    aggfunc="sum"
).round(2).rename(columns={"customer_lifetime_value": "total_revenue_proxy"})

print(revenue_by_sales_channel)
print(
    "\nHighest total revenue proxy channel:",
    revenue_by_sales_channel["total_revenue_proxy"].idxmax()
)

The channel with the largest total Customer Lifetime Value contributes the greatest aggregate customer value in this dataset. This should be interpreted as customer-value contribution rather than accounting revenue, because the file does not provide a direct revenue field.

### Exercise 2: Average customer lifetime value by gender and education

In [ ]:
average_clv_by_gender_education = marketing_data.pivot_table(
    index="education",
    columns="gender",
    values="customer_lifetime_value",
    aggfunc="mean"
).round(2)

print(average_clv_by_gender_education)
print("\nHighest gender-education combination:")
print(average_clv_by_gender_education.stack().idxmax())
print(average_clv_by_gender_education.stack().max().round(2))

This table makes it possible to compare customer value across education levels and genders. The highest cell identifies the segment with the greatest average lifetime value and can guide more targeted marketing.

## Bonus: Complaints by policy type and month

The sum of number_of_open_complaints is used because the column records the number of complaints per customer. The result is reset to a long-format DataFrame, with one policy type and month per row.

In [ ]:
complaints_by_policy_month = marketing_data.pivot_table(
    index=["policy_type", "month"],
    values="number_of_open_complaints",
    aggfunc="sum",
    fill_value=0
).reset_index()

complaints_by_policy_month = complaints_by_policy_month.rename(
    columns={"number_of_open_complaints": "total_complaints"}
).sort_values(["policy_type", "month"])

print(complaints_by_policy_month)

highest_complaint_months = complaints_by_policy_month.loc[
    complaints_by_policy_month.groupby("policy_type")["total_complaints"].idxmax()
]
print("\nHighest-complaint month for each policy type:")
print(highest_complaint_months)